### This notebook contains my final model attempt of getting the highest accuracy using retreival augmented fact checking (scraping the internet for facts)

##### I tried many different methods with BERT to try and train it and use it to classify misinformation however I hit a wall and could not improve the model any further. I know am going to attempt to make a retreival augmented model by utilizing wikipedia and DeBERTa-v3-base

In [ ]:
%pip install transformers accelerate datasets sentencepiece scikit-learn wikipedia

In [ ]:
import torch
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [7]:
# Load combined dataset
df = pd.read_csv("combinedClaimDataset.csv")

# Map labels to integers
label_map = {
    "supported": 0,
    "refuted": 1,
    "not_enough_info": 2
}

df["label"] = df["label"].map(label_map)
df = df.dropna(subset=["label", "claim", "source"])

# Downsample FEVER so it doesn't dominate
fever_df = df[df["source"] == "fever"]
other_df = df[df["source"] != "fever"]

max_fever = min(len(fever_df), 20000)  # cap FEVER at 20k (adjust if you want smaller)
fever_sampled = fever_df.sample(n=max_fever, random_state=42)

df_balanced = pd.concat([fever_sampled, other_df]).reset_index(drop=True)
df_balanced = df_balanced.sample(frac=1.0, random_state=42).reset_index(drop=True)  # shuffle
print(df_balanced["source"].value_counts())
print(df_balanced["label"].value_counts())

source
fever      20000
liar       12765
scifact      692
Name: count, dtype: int64
label
0    15516
2     9859
1     8082
Name: count, dtype: int64


In [8]:
train_df, temp_df = train_test_split(
    df_balanced,
    test_size=0.30,
    random_state=42,
    stratify=df_balanced["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

train_texts = train_df["claim"].tolist()
train_labels = train_df["label"].tolist()

val_texts = val_df["claim"].tolist()
val_labels = val_df["label"].tolist()

test_texts = test_df["claim"].tolist()
test_labels = test_df["label"].tolist()

len(train_texts), len(val_texts), len(test_texts)


(23419, 5019, 5019)

In [9]:
evidence_cache = {}

def retrieve_evidence(claim, max_snippets=2):
    if claim in evidence_cache:
        return evidence_cache[claim]
    try:
        results = wikipedia.search(claim)
        if not results:
            evidence_cache[claim] = ""
            return ""
        
        page = wikipedia.page(results[0], auto_suggest=False)
        summary = page.summary
        sentences = summary.split(". ")
        evidence = ". ".join(sentences[:max_snippets])
        evidence_cache[claim] = evidence
        return evidence
    except:
        evidence_cache[claim] = ""
        return ""

In [10]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

class EvidenceDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        claim = self.texts[idx]
        evidence = retrieve_evidence(claim)

        combined = f"Claim: {claim} Evidence: {evidence}"

        encoding = self.tokenizer(
            combined,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [11]:
train_dataset = EvidenceDataset(train_texts, train_labels, tokenizer)
val_dataset   = EvidenceDataset(val_texts,   val_labels,   tokenizer)
test_dataset  = EvidenceDataset(test_texts,  test_labels,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

In [12]:
device = torch.device("cpu")
torch.set_num_threads(4)

id2label = {0: "supported", 1: "refuted", 2: "not_enough_info"}
label2id = {"supported": 0, "refuted": 1, "not_enough_info": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
).to(device)

model = model.float()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias          

In [13]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=np.array(train_labels)
)
weights = torch.tensor(class_weights, dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

optimizer = AdamW(model.parameters(), lr=2e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)

In [14]:
def evaluate(model, dataloader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()
                     if k in ["input_ids", "attention_mask", "labels"]}

            outputs = model(**batch)
            logits = outputs.logits

            preds.extend(logits.argmax(dim=-1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    return f1_score(labels, preds, average="macro")

In [15]:
best_f1 = 0.0
patience = 2
patience_counter = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    model.train()

    progress = tqdm(train_loader)
    for batch in progress:
        batch = {k: v.to(device) for k, v in batch.items()
                 if k in ["input_ids", "attention_mask", "labels"]}

        outputs = model(**batch)
        loss = loss_fn(outputs.logits, batch["labels"])

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        progress.set_postfix({"loss": loss.item()})

    val_f1 = evaluate(model, val_loader)
    print(f"Validation Macro F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_ra_model.pt")
        print("Saved new best model.")
    else:
        patience_counter += 1
        print(f"No improvement. Patience {patience_counter}/{patience}")
        if patience_counter >= patience:
            print("Early stopping.")
            break


Epoch 1/3


  0%|          | 0/1464 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load("best_ra_model.pt", map_location=device))
test_f1 = evaluate(model, test_loader)
print(f"Test Macro F1: {test_f1:.4f}")